In [11]:
#imports
import numpy as np
import tensorflow as tf
import tensorflow_datasets as tfds
import keras

In [12]:
# load data
(ds_in, ds_out) = tfds.load('pneumonia_mnist', split=['train[:50%]', 'train[50%:100%]'], shuffle_files=True, as_supervised=True)

# Build your input pipeline
def normalize_img(image, label):
	"""Normalizes images: `uint8` -> `float32`."""
	return tf.cast(image, tf.float32) / 255., label

ds_in = ds_in.map(
    normalize_img, num_parallel_calls=tf.data.AUTOTUNE)
ds_in = ds_in.cache()
ds_in = ds_in.batch(64)
ds_in = ds_in.prefetch(tf.data.AUTOTUNE)


ds_out = ds_out.map(
    normalize_img, num_parallel_calls=tf.data.AUTOTUNE)
ds_out = ds_out.batch(64)
ds_out = ds_out.cache()
ds_out = ds_out.prefetch(tf.data.AUTOTUNE)

In [13]:
#target model and shadow model architecture
def get_classifier():
	model = tf.keras.models.Sequential([
	tf.keras.layers.Conv2D(128, (3, 3), activation="relu", input_shape=(28, 28, 1)),
	tf.keras.layers.MaxPool2D(2,2),
	tf.keras.layers.Conv2D(128, (3, 3), activation="relu"),
	tf.keras.layers.MaxPool2D(2,2),
	tf.keras.layers.Conv2D(32, (3, 3), activation="relu"),
	tf.keras.layers.MaxPool2D(2,2),
	tf.keras.layers.Flatten(),
	tf.keras.layers.Dense(64, activation='relu'),
	tf.keras.layers.Dense(10)
	])

	return model

In [14]:
#train target model on in data
target_model = get_classifier()

target_model.compile(
optimizer=tf.keras.optimizers.Adam(0.001),
loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
metrics=[tf.keras.metrics.SparseCategoricalAccuracy()],
)

target_model.fit(
    ds_in,
    epochs=25,
	validation_data=ds_out
)

Epoch 1/25


C:\Users\Skylar Marosi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


37/37 ━━━━━━━━━━━━━━━━━━━━ 2s 31ms/step - loss: 0.9715 - sparse_categorical_accuracy: 0.7205 - val_loss: 0.5573 - val_sparse_categorical_accuracy: 0.7447
Epoch 2/25
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - loss: 0.5549 - sparse_categorical_accuracy: 0.7396 - val_loss: 0.5307 - val_sparse_categorical_accuracy: 0.7447
Epoch 3/25
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - loss: 0.5196 - sparse_categorical_accuracy: 0.7417 - val_loss: 0.4702 - val_sparse_categorical_accuracy: 0.7451
Epoch 4/25
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - loss: 0.4228 - sparse_categorical_accuracy: 0.8008 - val_loss: 0.4292 - val_sparse_categorical_accuracy: 0.7787
Epoch 5/25
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - loss: 0.3447 - sparse_categorical_accuracy: 0.8530 - val_loss: 0.3564 - val_sparse_categorical_accuracy: 0.8326
Epoch 6/25
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - loss: 0.3208 - sparse_categorical_accuracy: 0.8636 - val_loss: 0.3262 - val_sparse_categorical_accuracy: 0.8619
Epoch 7/25
37/37 ━━━━

In [16]:
#train and save out-models using mixed loss criteria
mse = keras.losses.MeanSquaredError()
ce = keras.losses.SparseCategoricalCrossentropy()

for i in range(10):
	optimizer = tf.keras.optimizers.Adam(0.001)
	out_model = get_classifier()
	for epoch in range(25):
		for step, (x_batch_in, y_batch_in) in enumerate(ds_in):
			with tf.GradientTape() as tape:
				student_logits = out_model(x_batch_in, training=False)

				teacher_logits = target_model(x_batch_in)

				regular_loss = ce(y_batch_in, student_logits)

				imitate_loss = mse(teacher_logits, student_logits)

				loss_value = 0.5 * imitate_loss + 0.5 * regular_loss
				
			grads = tape.gradient(loss_value, out_model.trainable_weights)

			optimizer.apply_gradients(zip(grads, out_model.trainable_weights))
			
		print(f"completed epoch {epoch}")
	out_model.save_weights(f'./out_models/out_model_{i+1}.weights.h5')



completed epoch 0
completed epoch 1
completed epoch 2
completed epoch 3
completed epoch 4
completed epoch 5
completed epoch 6
completed epoch 7
completed epoch 8
completed epoch 9
completed epoch 10
completed epoch 11
completed epoch 12
completed epoch 13
completed epoch 14
completed epoch 15
completed epoch 16
completed epoch 17
completed epoch 18
completed epoch 19
completed epoch 20
completed epoch 21
completed epoch 22
completed epoch 23
completed epoch 24
completed epoch 0
completed epoch 1


KeyboardInterrupt: 

In [ ]:
#train and save in-models on in data
for i in range(10):
	in_model = get_classifier()

	in_model.compile(
	optimizer=tf.keras.optimizers.Adam(0.001),
	loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
	)

	in_model.fit(
    ds_in,
    epochs=25,
	)

	in_model.save_weights(f'./in_models/in_model_{i+1}.weights.h5')

In [ ]:
#for each query in D_query
# find real = phi(f(query))
# find each phi(f_out(query))
# find each phi(f_in(query))
# compute lambda = (real - mean(phi(f_out)))^2 - (real - mean(phi(f_in)))^2
# if lambda > 0 -> predict "in"